In [0]:
# Read from Bronze Delta table
df_bronze = spark.read \
    .format("delta") \
    .table("workspace.insurance_claims.bronze_claims_raw")

print("Records from Bronze:", df_bronze.count())
print("Columns:", df_bronze.columns)

Records from Bronze: 1000
Columns: ['months_as_customer', 'age', 'policy_number', 'policy_bind_date', 'policy_state', 'policy_csl', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'insured_relationship', 'capital-gains', 'capital-loss', 'incident_date', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state', 'incident_city', 'incident_location', 'incident_hour_of_the_day', 'number_of_vehicles_involved', 'property_damage', 'bodily_injuries', 'witnesses', 'police_report_available', 'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make', 'auto_model', 'auto_year', 'fraud_reported', 'ingestion_timestamp', 'source_file', 'layer']


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Check nulls in every column
null_counts = df_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

null_counts.show(truncate=False)

+------------------+---+-------------+----------------+------------+----------+-----------------+---------------------+--------------+-----------+-----------+-----------------------+------------------+---------------+--------------------+-------------+------------+-------------+-------------+--------------+-----------------+---------------------+--------------+-------------+-----------------+------------------------+---------------------------+---------------+---------------+---------+-----------------------+------------------+------------+--------------+-------------+---------+----------+---------+--------------+-------------------+-----------+-----+
|months_as_customer|age|policy_number|policy_bind_date|policy_state|policy_csl|policy_deductable|policy_annual_premium|umbrella_limit|insured_zip|insured_sex|insured_education_level|insured_occupation|insured_hobbies|insured_relationship|capital-gains|capital-loss|incident_date|incident_type|collision_type|incident_severity|authorities_co

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Show nulls vertically - much easier to read
null_counts = df_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

# Transpose to vertical view
import pandas as pd
null_pandas = null_counts.toPandas().T
null_pandas.columns = ["null_count"]
null_pandas = null_pandas[null_pandas["null_count"] > 0]

if len(null_pandas) == 0:
    print("✅ No nulls found in any column!")
else:
    print("⚠️ Columns with nulls:")
    print(null_pandas)

✅ No nulls found in any column!


In [0]:
from pyspark.sql.functions import count

# Total records
total = df_bronze.count()

# Distinct records
distinct = df_bronze.distinct().count()

duplicates = total - distinct

print(f"Total Records: {total}")
print(f"Distinct Records: {distinct}")
print(f"Duplicate Records: {duplicates}")

if duplicates == 0:
    print("✅ No duplicates found!")
else:
    print(f"⚠️ {duplicates} duplicates found — will be removed")

Total Records: 1000
Distinct Records: 1000
Duplicate Records: 0
✅ No duplicates found!


In [0]:
from pyspark.sql.functions import (
    col, when, to_date, year, month, 
    datediff, current_date, upper, trim
)

df_silver = df_bronze

# 1. Standardize date columns
df_silver = df_silver \
    .withColumn("incident_date_clean",
        to_date(col("incident_date"), "yyyy-MM-dd")) \
    .withColumn("policy_bind_date_clean",
        to_date(col("policy_bind_date"), "yyyy-MM-dd"))

# 2. Extract year and month from incident date
df_silver = df_silver \
    .withColumn("incident_year",
        year(col("incident_date_clean"))) \
    .withColumn("incident_month",
        month(col("incident_date_clean")))

# 3. Add age band
df_silver = df_silver \
    .withColumn("age_band",
        when(col("age") < 25, "Young")
        .when(col("age") < 40, "Young Adult")
        .when(col("age") < 55, "Middle Aged")
        .otherwise("Senior"))

# 4. Add claim severity band
df_silver = df_silver \
    .withColumn("claim_severity_band",
        when(col("total_claim_amount") < 10000, "Low")
        .when(col("total_claim_amount") < 40000, "Medium")
        .when(col("total_claim_amount") < 80000, "High")
        .otherwise("Critical"))

# 5. Standardize text columns
df_silver = df_silver \
    .withColumn("insured_sex",
        upper(trim(col("insured_sex")))) \
    .withColumn("incident_severity",
        upper(trim(col("incident_severity"))))

# 6. Add fraud flag as integer
df_silver = df_silver \
    .withColumn("is_fraud",
        when(col("fraud_reported") == "Y", 1)
        .otherwise(0))

# 7. Calculate policy age in days
df_silver = df_silver \
    .withColumn("policy_age_days",
        datediff(
            col("incident_date_clean"),
            col("policy_bind_date_clean")))

# Drop original raw date columns and audit cols
df_silver = df_silver \
    .drop("incident_date") \
    .drop("policy_bind_date") \
    .drop("layer") \
    .drop("source_file") \
    .drop("ingestion_timestamp")

# Preview
print("Silver columns:", len(df_silver.columns))
df_silver.show(5, truncate=False)

Silver columns: 45
+------------------+---+-------------+------------+----------+-----------------+---------------------+--------------+-----------+-----------+-----------------------+------------------+---------------+--------------------+-------------+------------+------------------------+---------------+-----------------+---------------------+--------------+-------------+-----------------+------------------------+---------------------------+---------------+---------------+---------+-----------------------+------------------+------------+--------------+-------------+---------+----------+---------+--------------+-------------------+----------------------+-------------+--------------+-----------+-------------------+--------+---------------+
|months_as_customer|age|policy_number|policy_state|policy_csl|policy_deductable|policy_annual_premium|umbrella_limit|insured_zip|insured_sex|insured_education_level|insured_occupation|insured_hobbies|insured_relationship|capital-gains|capital-loss|i

In [0]:
# Write Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.insurance_claims.silver_claims_cleansed")

print("✅ Silver Delta table written successfully!")
print("Total rows:", df_silver.count())
print("Total columns:", len(df_silver.columns))

✅ Silver Delta table written successfully!
Total rows: 1000
Total columns: 45
